# CNET Phase III — Colab runner (scaffold)

Thin launcher for code that lives in the repository (`src/`). **No scientific code belongs in these cells.**
See `docs/COLAB_EXECUTION_PLAN.md`.

**Status:** scaffold only — **not yet run on Colab**. Cells that touch Google Drive or datasets are
disabled until **D-015 Amendment 1** (see `docs/decision_log.md`) authorizes processing data on Google
infrastructure. Free-tier Colab may not assign a GPU and may disconnect; outputs must go to persistent
storage (plan §7).

## 1. Runtime check
Runtime → Change runtime type → GPU (not guaranteed on free tier).

In [ ]:
!nvidia-smi || echo "No NVIDIA GPU attached to this runtime"
!python --version

## 2. Get the code at an exact commit
Set `REPO_URL` once the GitHub remote exists, and `COMMIT` to the hash Claude reports. For a private repository, store a read-only token in Colab Secrets as `GITHUB_TOKEN` (never paste it here).

In [ ]:
REPO_URL = ""   # e.g. "https://github.com/<user>/<repo>.git" — set after the remote is configured
COMMIT = ""     # exact commit to run; recorded in every run
assert REPO_URL and COMMIT, "Set REPO_URL and COMMIT first"

import os
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")  # only needed for a private repository
    if token:
        url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
except Exception:
    pass
if not os.path.isdir("/content/cnet"):
    !git clone -q {url} /content/cnet
%cd /content/cnet
!git checkout -q {COMMIT} && git log -1 --oneline

## 3. Dependencies and tests on this runtime
Uses Colab's preinstalled numpy/pandas/scikit-learn/torch; installs only what is missing (plan §1). The test suite must pass here before any experiment.

In [ ]:
!pip install -q pytest
!python -m pytest -q -rs
!python -m src.env

## 4. Persistent storage and data — DISABLED until D-015 Amendment 1
Uncomment only after the amendment is recorded. The archive is copied to the VM's local disk, verified against the manifest (size, sha256, CIC MD5) and extracted locally (plan §5–§7).

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")                       # you approve Google's prompt
# DATA = "/content/drive/MyDrive/cnet-data"            # private, unshared folder (amendment)
# RUNS = "/content/drive/MyDrive/cnet-runs"
# !mkdir -p data/raw/cicids2017_original {RUNS}
# !cp "{DATA}/cicids2017_original/MachineLearningCSV.zip" data/raw/cicids2017_original/
# !python -m src.data.acquire verify cicids2017_original
# !python -m src.data.acquire extract cicids2017_original
# !rm -rf results/runs && ln -s "{RUNS}" results/runs   # run outputs survive disconnection
# !python -m pytest -q tests/test_real_data_smoke.py -k friday

## 5. Experiment — Milestone 1 (not implemented yet)
The command below does not exist yet; it will be a module in `src/` with resume support (plan §8).

In [ ]:
# !python -m src.experiments.<name> --config configs/experiments/<experiment>.yaml --device cuda